# RQ2 — Does generative LLM uncertainty rise as much for events experts contest as for events the model mislabels?

This notebook reproduces **§3.2 (RQ2)** of the paper: Tables 5 and 6, the within-category and
model-independent checks, and the robustness analyses in **Appendix C** (per-model results, types of
contestation, subsets and the passage-level analysis). It also reproduces the measurement details in
**§2.5** (letter-token coverage) and looks up the model outputs behind the close-read examples in
**§3.3**.

All code is in the cells below. Inputs: the Label Studio exports and the stored model scores in `ls_real/`.
Run the cells from top to bottom; the two bootstraps in Step 3 take a few minutes.

**Contents**
0. From Label Studio exports to readings; the model scores
1. Measurement details (§2.5)
2. The three groups (Table 5)
3. AUROC with a text-clustered bootstrap (Table 6, Appendix Table *RQ2 per model*)
4. Robustness checks (Appendix C)
5. Model outputs for the close-read examples (§3.3)
6. Recap

In [1]:
import os, json, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

# The data lives in ls_real/ at the root of the redress_paper folder. Walk up from the
# notebook's folder until we find it, and work from there so that all paths are relative.
REPO = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(REPO, 'ls_real')):
    parent = os.path.dirname(REPO)
    if parent == REPO:
        raise FileNotFoundError('Run this notebook from inside the redress_paper folder (ls_real/ not found).')
    REPO = parent
os.chdir(REPO)
D = 'ls_real'
FIG_DIR = os.path.join('rq_notebooks', 'figures')
os.makedirs(FIG_DIR, exist_ok=True)
pd.set_option('display.width', 250)
pd.set_option('display.max_columns', 40)
pd.set_option('display.max_colwidth', 60)
print('working directory:', REPO)

working directory: c:\Users\tessd\RedressHub_paper


## Step 0 — from Label Studio exports to *readings*

The double-annotated dataset is stored as pairs of Label Studio exports in `ls_real/`. Each pair
covers the same texts, annotated independently by two annotators (a *pool*):

| pool | annotator A | annotator B | pair |
|---|---|---|---|
| `iaa` | Cira (re-annotation) | Elke (re-annotation) | domain experts |
| `cira_solo` | Cira (hesitancy added retrospectively) | Elke (in the moment) | domain experts |
| `elke_solo` | Elke (hesitancy added retrospectively) | Cira (in the moment) | domain experts |
| `iaa_nl` | Elke (re-annotation) | trained annotator A3 | expert × trained |
| `elke_solo_nl` | Elke (hesitancy added retrospectively) | trained annotator A3 | expert × trained |

The code below turns every labelled span into one **reading** (paper, §1.1): one annotator's label on
one span. For each reading we look for the partner annotator's spans that overlap it by at least one
character:

* `other_label` — the label of the partner's span with the largest overlap (empty if the partner marked
  nothing there: a *one-sided* span);
* `divergent` — 1 if none of the overlapping partner spans has the same label;
* `friction` — 1 if the reading is divergent **or** the annotator rated it *somewhat* or *very hesitant*;
* `ftype` — `agreed` (no friction), `contested` (friction, and the partner marked the passage too) or
  `detection` (only this annotator marked the passage).

Three spans were accidentally labelled twice (same span, same label) by the same annotator; we keep one
copy of each (`dedup=True`). Two other spans deliberately carry two different labels (an event type and
`Organisation`); both readings are kept. One Dutch article was annotated twice, in two different pools;
we keep one copy (`EXCLUDE_TEXTS`, see the duplicate check in the RQ1 notebook). The three vaguer *Time*
labels are left out of the analysis, as in the paper.

**No article texts.** The articles remain under their publishers' copyright, so the exports contain only
character offsets, no text. `ls_real/texts.csv` lists, for every text, its source URL, source, language,
length and a SHA-256 checksum of the exact text that was annotated. None of the analyses needs the texts
themselves. If the full texts are available locally (project members: see the README), `load_texts()`
returns them, and the notebooks then also show the span texts and re-run the duplicate check from scratch.

In [2]:
RATING = {'certain': 0, 'mostly certain': 1, 'somewhat hesitant': 2, 'very hesitant': 3}
NAME = {1182: 'Cira', 1188: 'Elke', 644: 'A3'}             # Label Studio user id -> annotator
POOLS = [  # pool, export of annotator A, export of annotator B (same texts), annotator pair
    ('iaa', 'export_reann_cira.json', 'export_reann_elke.json', 'expert-expert'),
    ('cira_solo', 'export_hesretro_cira.json', 'export_cross_elke.json', 'expert-expert'),
    ('elke_solo', 'export_hesretro_elke.json', 'export_cross_cira.json', 'expert-expert'),
    ('iaa_nl', 'export_reann_elke_nl.json', 'export_reann_a3.json', 'expert-trained'),
    ('elke_solo_nl', 'export_hesretro_elke_nl.json', 'export_cross_a3.json', 'expert-trained'),
]
# descriptive labels; every other label is normative
STRATUM = {'Organisation': 'descriptive', 'Time_absolute': 'descriptive',
           'Books': 'descriptive', 'Location': 'descriptive'}
DROP = {'Time_relative', 'Time_range', 'Time_vague'}         # left out of the analysis
KEYS = ['text_key', 'annotator', 'start', 'end']              # identifies one reading
# The same Dutch article was annotated twice, once in the iaa_nl pool and once in the elke_solo_nl pool
# (see the duplicate check in the RQ1 notebook). We keep the iaa_nl copy, in which both annotators rated
# their hesitancy while annotating, and leave out the other copy.
EXCLUDE_TEXTS = {'elke_solo_nl:335223'}
# one row per text: source URL, source, language, length and SHA-256 checksum of the annotated text
TEXT_META = pd.read_csv(os.path.join(D, 'texts.csv')).set_index('text_key')


def load_texts():
    """The full annotated texts, if available locally (ls_real/texts_full.json, or exports that still contain
    the texts); otherwise None. Each text is checked against its checksum in texts.csv."""
    import hashlib
    path = os.path.join(D, 'texts_full.json')
    texts = json.load(open(path, encoding='utf-8')) if os.path.exists(path) else {}
    if not texts:
        for pool, fa, fb, pair in POOLS:
            for f in (fa, fb):
                for t in json.load(open(os.path.join(D, f), encoding='utf-8')):
                    if 'text' in t['data']:
                        texts[f'{pool}:{t["data"]["orig_task_id"]}'] = t['data']['text']
    for k, v in texts.items():
        assert hashlib.sha256(v.encode('utf-8')).hexdigest() == TEXT_META.loc[k, 'sha256'], f'{k}: checksum mismatch'
    return texts or None


def parse(task, drop, collapse):
    """Spans and per-span choices (hesitancy, alternative label, source of doubt) of one annotation."""
    canon = lambda label: collapse.get(label, label)
    ann = [a for a in task['annotations'] if not a.get('was_cancelled')][0]
    spans = [dict(id=r['id'], start=r['value']['start'], end=r['value']['end'], label=canon(r['value']['labels'][0]))
             for r in ann['result'] if r.get('type') == 'labels' and canon(r['value']['labels'][0]) not in drop]
    choices = {'hesitancy': {}, 'alt_label': {}, 'doubt_source': {}}
    for r in ann['result']:
        if r.get('type') == 'choices' and r.get('from_name') in choices:
            v = r['value']['choices'][0]
            if r['from_name'] == 'alt_label':
                v = canon(v)
                if v in drop:
                    continue
            choices[r['from_name']][r['id']] = v
    return ann['completed_by'], spans, choices


def overlap(a, b):
    """Two spans overlap if they share at least one character."""
    return a['start'] < b['end'] and b['start'] < a['end']


def rows_for(task, other_task, pool, pair, drop, collapse, stratum):
    """One row (reading) per span of `task`, compared with the partner's annotation `other_task`."""
    user, spans, choices = parse(task, drop, collapse)
    _, other_spans, _ = parse(other_task, drop, collapse)
    text_key = f'{pool}:{task["data"]["orig_task_id"]}'
    mode = 'retrospective' if task['data'].get('arm') == 'retrospective' else 'in-the-moment'
    out = []
    for s in spans:
        cands = [o for o in other_spans if overlap(s, o)]
        matched = any(o['label'] == s['label'] for o in cands)
        other_label = (max(cands, key=lambda o: min(s['end'], o['end']) - max(s['start'], o['start']))['label']
                       if cands else None)
        out.append(dict(pool=pool, pair=pair, text_key=text_key,
                        annotator=NAME.get(user, user), mode=mode, lang=TEXT_META.loc[text_key, 'lang'],
                        start=s['start'], end=s['end'], label=s['label'],
                        stratum=stratum.get(s['label'], 'normative'),
                        hesitancy=RATING[choices['hesitancy'][s['id']]],
                        alt_label=choices['alt_label'].get(s['id']),
                        doubt_source=choices['doubt_source'].get(s['id']),
                        other_label=other_label, divergent=int(not matched)))
    return out


def build_readings(drop=DROP, collapse=None, stratum=STRATUM, dedup=True, exclude=EXCLUDE_TEXTS):
    collapse = collapse or {}
    rows = []
    for pool, fa, fb, pair in POOLS:
        a = {t['data']['orig_task_id']: t for t in json.load(open(os.path.join(D, fa), encoding='utf-8'))}
        b = {t['data']['orig_task_id']: t for t in json.load(open(os.path.join(D, fb), encoding='utf-8'))}
        assert set(a) == set(b), f'{pool}: the two exports do not cover the same texts'
        for k in a:
            if f'{pool}:{k}' in exclude:
                continue
            rows += rows_for(a[k], b[k], pool, pair, drop, collapse, stratum)
            rows += rows_for(b[k], a[k], pool, pair, drop, collapse, stratum)
    df = pd.DataFrame(rows)
    df['friction'] = ((df.divergent == 1) | (df.hesitancy >= 2)).astype(int)
    if dedup:  # three spans were accidentally labelled twice (same span, same label) by the same annotator
        df = df.drop_duplicates(KEYS + ['label']).reset_index(drop=True)
    df['ftype'] = np.where(df.friction == 0, 'agreed', np.where(df.other_label.notna(), 'contested', 'detection'))
    # 'first' marks the readings of one (alphabetically first) annotator per text, so that each
    # label split is counted once rather than once per annotator
    who = df.groupby('text_key').annotator.agg(lambda s: sorted(set(s)))
    df['first'] = [a == who[k][0] for k, a in zip(df.text_key, df.annotator)]
    return df


def passages(frame):
    """Passage id per reading: readings whose spans overlap (directly or via a chain) share a passage."""
    pid = np.full(len(frame), -1)
    nxt = 0
    for _, g in frame.groupby('text_key'):
        idx = g.index.values
        s, e = g.start.values, g.end.values
        parent = list(range(len(idx)))
        def find(i):
            while parent[i] != i:
                parent[i] = parent[parent[i]]
                i = parent[i]
            return i
        for i in range(len(idx)):
            for j in range(i + 1, len(idx)):
                if s[i] < e[j] and s[j] < e[i]:
                    parent[find(i)] = find(j)
        roots = {}
        for i in range(len(idx)):
            r = find(i)
            if r not in roots:
                roots[r] = nxt
                nxt += 1
            pid[frame.index.get_loc(idx[i])] = roots[r]
    return pid

In [3]:
df = build_readings()
df['pid'] = passages(df)
print(f'{len(df)} readings in {df.text_key.nunique()} texts')
df[['pool', 'pair', 'text_key', 'annotator', 'label', 'stratum', 'hesitancy', 'other_label', 'divergent', 'ftype']].head(8)

1469 readings in 67 texts


,pool,pair,text_key,annotator,label,stratum,hesitancy,other_label,divergent,ftype
0,iaa,expert-expert,iaa:285647,Cira,Truth-seeking initiatives,normative,0,Truth-seeking initiatives,0,agreed
1,iaa,expert-expert,iaa:285647,Cira,Truth-seeking initiatives,normative,0,Truth-seeking initiatives,0,agreed
2,iaa,expert-expert,iaa:285647,Cira,Time_absolute,descriptive,0,None,1,detection
3,iaa,expert-expert,iaa:285647,Cira,Public acknowledgements,normative,0,Apologies,1,contested
4,iaa,expert-expert,iaa:285647,Cira,Organisation,descriptive,0,Organisation,0,agreed
5,iaa,expert-expert,iaa:285647,Cira,Time_absolute,descriptive,0,None,1,detection
6,iaa,expert-expert,iaa:285647,Cira,Time_absolute,descriptive,2,None,1,detection
7,iaa,expert-expert,iaa:285647,Cira,Truth-seeking initiatives,normative,2,Truth-seeking initiatives,0,contested


### The model scores

The LLM outputs were computed once and are stored as CSV files in `ls_real/` (one row per reading;
re-running the API calls is not needed to reproduce the analysis). For each of the six models there is
one file per uncertainty measure:

* `entropy` — the **doubt score** used in RQ2 (normalised entropy of the label distribution; for the
  verbal measure, 1 − mean stated confidence);
* `consistency` — the **confidence** used for thresholds in RQ3 (for log-probabilities: the expected share
  of the majority answer when drawing ten answers at temperature 0.7 from the model's distribution; for
  sampled answers: the share of the majority answer among the ten samples; for verbal: the mean stated
  confidence);
* `pred_label` — the model's answer.

**Note on the log-probability confidence.** The stored `consistency` column of the log-probability files
was computed per prompt template and then averaged over the three templates. If two templates confidently
disagree, that average stays close to 1, although the model is clearly unsure. The paper (§2.5) defines the
confidence on the label distribution *averaged* over the templates, as are the model's answer and the doubt
score. The function `read_scores` below therefore recomputes the log-probability confidence from the stored
averaged distribution (`p_<label>` columns), with the same simulation as `phase3_llm.py`. The stored value is
kept as `consistency_stored`.

In [4]:
FAM = {  # model -> (log-probability file, sampled file, verbal file)
    'gpt-4o-mini': ('machine_scores.csv', 'machine_scores_sampled_mini.csv', 'machine_scores_verbal_mini.csv'),
    'gpt-4o': ('machine_scores_gpt4o.csv', 'machine_scores_sampled_gpt4o.csv', 'machine_scores_verbal_gpt4o.csv'),
    'qwen3': ('machine_scores_qwen3.csv', 'machine_scores_sampled_qwen3.csv', 'machine_scores_verbal_qwen3.csv'),
    'llama-3.3': ('machine_scores_llama.csv', 'machine_scores_sampled_llama.csv', 'machine_scores_verbal_llama.csv'),
    'mistral-3.2': ('machine_scores_mistral.csv', 'machine_scores_sampled_mistral.csv', 'machine_scores_verbal_mistral.csv'),
    'gemma-3': ('machine_scores_gemma.csv', 'machine_scores_sampled_gemma.csv', 'machine_scores_verbal_gemma.csv'),
}
MEASURES = ('logprob', 'sampled', 'verbal')
CONT = 'Contestation/mobilisation'
NONE = 'None of these / not redress-related'
ARG = {'Organisation', 'Location', 'Time_absolute', 'Time_range', 'Time_relative', 'Time_vague'}


def consistency_at_temp(p, k=10, temp=0.7, reps=2000, seed=0):
    """Expected share of the majority answer among k answers drawn at temperature `temp` from distribution p
    (simulated; the same function as in phase3_llm.py)."""
    q = np.power(np.clip(p, 1e-12, 1), 1 / temp)
    q /= q.sum()
    draws = np.random.default_rng(seed).multinomial(k, q, size=reps)
    return float(draws.max(axis=1).mean() / k)


_LP_CONF = {}
def read_scores(fam, meas):
    """All stored scores of one run; for log-probabilities, the confidence is recomputed from the
    template-averaged label distribution (see the note above)."""
    f = FAM[fam][MEASURES.index(meas)]
    m = pd.read_csv(os.path.join(D, f))
    if meas == 'logprob':
        if f not in _LP_CONF:
            lab = [c for c in m.columns if c.startswith('p_') and c != 'p_none']
            _LP_CONF[f] = np.array([consistency_at_temp(p / p.sum()) for p in m[lab].values])
        m['consistency_stored'] = m['consistency']
        m['consistency'] = _LP_CONF[f]
    return m


def scores(fam, meas, cols=('pred_label', 'consistency', 'entropy')):
    return read_scores(fam, meas)[KEYS + list(cols)].drop_duplicates(KEYS)

## Step 1 — Measurement details (§2.5)

For the log-probability measure, every reading is scored with three prompt templates; for each call we
read the probability of the 23 option letters among the 20 most likely tokens. The share of probability
those letters take up (`letter_mass`) shows how much the renormalisation changes. We also check that the
verbal doubt score is exactly 1 − mean stated confidence.

In [5]:
TEMPLATES = {'gpt-4o-mini': 'machine_scores_templates.csv', 'gpt-4o': 'machine_scores_templates_gpt4o.csv',
             'qwen3': 'machine_scores_templates_qwen3.csv', 'llama-3.3': 'machine_scores_templates_llama.csv',
             'mistral-3.2': 'machine_scores_templates_mistral.csv', 'gemma-3': 'machine_scores_templates_gemma.csv'}
base_keys = df[(df.stratum == 'normative') & df.ftype.isin(['agreed', 'contested'])][KEYS]
rows = []
for fam, f in TEMPLATES.items():
    t = pd.read_csv(os.path.join(D, f)).merge(base_keys, on=KEYS)
    rows.append({'model': fam, 'template calls': len(t), 'mean letter mass': f'{t.letter_mass.mean():.3f}',
                 'calls with mass < .90': f'{(t.letter_mass < .9).mean():.1%}'})
display(pd.DataFrame(rows))

v = pd.read_csv(os.path.join(D, FAM['gpt-4o-mini'][2]))
print('verbal: max |entropy - (1 - verbal confidence)| =', (v.entropy - (1 - v.verbal)).abs().max())

,model,template calls,mean letter mass,calls with mass < .90
0,gpt-4o-mini,1095,1.000,0.1%
1,gpt-4o,1095,1.000,0.0%
2,qwen3,1095,0.999,0.4%
3,llama-3.3,1095,1.000,0.0%
4,mistral-3.2,1095,0.928,30.5%
5,gemma-3,1095,1.000,0.0%


verbal: max |entropy - (1 - verbal confidence)| = 1.6653345369377348e-16


## Step 2 — The three groups (Table 5)

The population is every reading of a normative span that both annotators marked. Contested readings are
those with friction (a different label from the partner, or a hesitant rating). The agreed readings are
split per model run: **easy** if the model gives the agreed label, **mislabelled** otherwise.

`group_indep` is the model-independent version used in §3.2: an agreed reading counts as mislabelled if
the majority of the *other five* models (log-probabilities) get it wrong, and as easy otherwise.

In [6]:
def load_runs(df):
    base = df[(df.stratum == 'normative') & df.ftype.isin(['contested', 'agreed'])]
    preds = base[KEYS].copy()
    for fam, (f, _, _) in FAM.items():
        preds = preds.merge(pd.read_csv(os.path.join(D, f))[KEYS + ['pred_label']].rename(columns={'pred_label': fam}),
                            on=KEYS, how='left')
    runs = []
    for fam, files in FAM.items():
        others = [o for o in FAM if o != fam]
        for meas, f in zip(MEASURES, files):
            m = pd.read_csv(os.path.join(D, f))[KEYS + ['entropy', 'pred_label']].rename(columns={'entropy': 'doubt'})
            x = base.merge(m, on=KEYS).merge(preds, on=KEYS).reset_index(drop=True)
            x['group'] = np.where(x.ftype == 'contested', 'contested',
                                  np.where(x.pred_label == x.label, 'easy', 'mislabelled'))
            share = np.mean([x[o] == x.label for o in others], axis=0)
            x['group_indep'] = np.where(x.ftype == 'contested', 'contested', np.where(share >= .5, 'easy', 'mislabelled'))
            runs.append((fam, meas, x))
    return runs

runs = load_runs(df)
counts = pd.DataFrame([{'model': f, 'measure': m, **x.group.value_counts().to_dict()} for f, m, x in runs])
display(counts[counts.measure == 'logprob'])
lp = counts[counts.measure == 'logprob']
print(f"Table 5 (log-probability runs): easy {lp.easy.min()}-{lp.easy.max()}, mislabelled {lp.mislabelled.min()}-"
      f"{lp.mislabelled.max()}, together {int((lp.easy + lp.mislabelled).iloc[0])} agreed; contested {int(lp.contested.iloc[0])}")

,model,measure,easy,contested,mislabelled
0,gpt-4o-mini,logprob,181,124,60
3,gpt-4o,logprob,212,124,29
6,qwen3,logprob,211,124,30
9,llama-3.3,logprob,185,124,56
12,mistral-3.2,logprob,202,124,39
15,gemma-3,logprob,197,124,44


Table 5 (log-probability runs): easy 181-212, mislabelled 29-60, together 241 agreed; contested 124


## Step 3 — AUROC with a text-clustered bootstrap

The AUROC is the probability that a randomly chosen reading from the harder group gets more doubt than a
randomly chosen easy reading (.50 = coin flip). It equals the rescaled Mann–Whitney U statistic and is
computed from ranks. `auc_within` computes it within each label and pools the pairs, so that differences
between categories cannot drive it.

For every model × measure run (`pair_stats`) we compute:
* `C` contested vs. easy, `M` mislabelled vs. easy, `M-C` their difference;
* `C within`, `M within` the within-label versions;
* `C indep`, `M indep`, `M-C indep` with the model-independent groups.

`analyse` reports these per run and averaged over the six models per measure. Confidence intervals come
from 1,000 bootstrap samples that resample whole **texts** (all runs share each resample), so that readings
from the same text, including the two readings of the same passage, are never treated as independent.

In [7]:
from scipy.stats import rankdata

STATS = ['C', 'M', 'M-C', 'M vs C', 'C within', 'M within', 'C indep', 'M indep', 'M-C indep']
REPS = 1000


def auc(s, y):
    n1 = y.sum(); n0 = len(y) - n1
    if not n1 or not n0:
        return np.nan
    return (rankdata(s)[y].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)


def auc_within(s, y, lab):
    U = P = 0
    for l in np.unique(lab):
        k = lab == l
        yk = y[k]; n1 = yk.sum(); n0 = k.sum() - n1
        if n1 and n0:
            U += rankdata(s[k])[yk].sum() - n1 * (n1 + 1) / 2; P += n1 * n0
    return U / P if P else np.nan


def pair_stats(x):
    s, g, lab, gi = x.doubt.values, x.group.values, x.label.values, x.group_indep.values
    out = {}
    for tag, grp in [('', g), (' indep', gi)]:
        for key, pos in [('C', 'contested'), ('M', 'mislabelled')]:
            k = np.isin(grp, [pos, 'easy'])
            if tag:
                out[key + tag] = auc(s[k], grp[k] == pos)
            else:
                out[key] = auc(s[k], grp[k] == pos)
                out[key + ' within'] = auc_within(s[k], grp[k] == pos, lab[k])
        out['M-C' + tag] = out['M' + tag] - out['C' + tag]
    k = np.isin(g, ['mislabelled', 'contested'])
    out['M vs C'] = auc(s[k], g[k] == 'mislabelled')
    return out


def analyse(runs, rng):
    """Point estimates per run and a joint text-clustered bootstrap; returns one row per unit x statistic."""
    texts = sorted(set().union(*[set(x.text_key) for _, _, x in runs]))
    pos = [{t: g.index.values for t, g in x.groupby('text_key')} for _, _, x in runs]
    point = np.array([[pair_stats(x)[k] for k in STATS] for _, _, x in runs])
    B = np.full((REPS, len(runs), len(STATS)), np.nan)
    for r in range(REPS):
        draw = rng.choice(texts, len(texts))
        for i, (_, _, x) in enumerate(runs):
            idx = np.concatenate([pos[i][t] for t in draw if t in pos[i]])
            st = pair_stats(x.loc[idx])
            B[r, i] = [st[k] for k in STATS]
    ci = lambda a: (np.nanpercentile(a, 2.5), np.nanpercentile(a, 97.5))
    names = [f'{fam} {meas}' for fam, meas, _ in runs]
    groups = {f'mean {m}': [i for i, (_, mm, _) in enumerate(runs) if mm == m] for m in MEASURES}
    summ = []
    for unit, ii in [(names[i], [i]) for i in range(len(runs))] + list(groups.items()):
        for j, k in enumerate(STATS):
            lo, hi = ci(np.nanmean(B[:, ii, j], axis=1))
            summ.append({'unit': unit, 'stat': k, 'est': np.nanmean(point[ii, j]), 'lo': lo, 'hi': hi})
    return pd.DataFrame(summ)


def show(summary, units, stats_):
    s = summary[summary.unit.isin(units) & summary.stat.isin(stats_)].copy()
    s['value'] = [f'{e:.2f} [{l:.2f}, {h:.2f}]' for e, l, h in zip(s.est, s.lo, s.hi)]
    return s.pivot(index='unit', columns='stat', values='value').reindex(index=units, columns=stats_)

In [8]:
%%time
rng = np.random.default_rng(20260929)    # one generator for both bootstraps, in this order
summary_all = analyse(runs, rng)
runs_experts = [(f, m, x[x.pair == 'expert-expert'].reset_index(drop=True)) for f, m, x in runs]
summary_experts = analyse(runs_experts, rng)

CPU times: total: 2min 57s
Wall time: 3min 8s


### Table 6 — mean AUROC over the six models, per measure

`C`: contested vs. easy; `M`: mislabelled vs. easy; `M-C`: the gap. The upper bound of the confidence
interval for `C` (log-probabilities: .72) is what §3.2 calls the best case.

In [9]:
show(summary_all, ['mean logprob', 'mean sampled', 'mean verbal'], ['C', 'M', 'M-C'])

stat,C,M,M-C
unit,,,
mean logprob,"0.66 [0.58, 0.72]","0.79 [0.73, 0.87]","0.13 [0.06, 0.20]"
mean sampled,"0.62 [0.57, 0.67]","0.72 [0.66, 0.80]","0.10 [0.04, 0.17]"
mean verbal,"0.56 [0.49, 0.63]","0.45 [0.36, 0.54]","-0.12 [-0.20, -0.02]"


The last row of Table 6 compares contested readings with **all** agreed readings, whether or not the
model gets them right (the fairest comparison for a deployed pipeline, which cannot tell easy from
mislabelled readings). It uses its own bootstrap.

In [10]:
rng = np.random.default_rng(20260929)
texts = sorted(set().union(*[set(x.text_key) for _, _, x in runs]))
for meas in MEASURES:
    R = [x for f, m, x in runs if m == meas]
    per = [auc(x.doubt.values, (x.group == 'contested').values) for x in R]
    pos = [{t: np.flatnonzero(x.text_key.values == t) for t in set(x.text_key)} for x in R]
    B = []
    for _ in range(1000):
        draw = rng.choice(texts, len(texts))
        v = []
        for x, p in zip(R, pos):
            idx = np.concatenate([p[t] for t in draw if t in p])
            v.append(auc(x.doubt.values[idx], (x.group.values[idx] == 'contested')))
        B.append(np.nanmean(v))
    print(f'contested vs. all agreed, {meas:8s}: {np.mean(per):.2f} [{np.percentile(B, 2.5):.2f}, {np.percentile(B, 97.5):.2f}]'
          f'   (models {min(per):.2f}-{max(per):.2f})')

contested vs. all agreed, logprob : 0.61 [0.53, 0.68]   (models 0.57-0.67)


contested vs. all agreed, sampled : 0.58 [0.53, 0.63]   (models 0.55-0.63)


contested vs. all agreed, verbal  : 0.57 [0.50, 0.63]   (models 0.51-0.63)


### Within a category, and with model-independent groups (§3.2)

`C within`: contested vs. easy, compared only within the same label. `M-C indep`: the gap when easy and
mislabelled are defined by the other five models.

In [11]:
show(summary_all, ['mean logprob', 'mean sampled', 'mean verbal'], ['C', 'C within', 'M within', 'C indep', 'M indep', 'M-C indep'])

stat,C,C within,M within,C indep,M indep,M-C indep
unit,,,,,,
mean logprob,"0.66 [0.58, 0.72]","0.57 [0.49, 0.65]","0.74 [0.59, 0.89]","0.65 [0.57, 0.71]","0.74 [0.67, 0.82]","0.09 [0.01, 0.18]"
mean sampled,"0.62 [0.57, 0.67]","0.55 [0.50, 0.60]","0.67 [0.56, 0.81]","0.61 [0.55, 0.66]","0.66 [0.58, 0.74]","0.05 [-0.01, 0.13]"
mean verbal,"0.56 [0.49, 0.63]","0.54 [0.48, 0.61]","0.49 [0.37, 0.62]","0.58 [0.50, 0.64]","0.49 [0.38, 0.62]","-0.09 [-0.20, 0.04]"


### Per model (Appendix C, Table *RQ2 per model*) and the two domain experts alone

In [12]:
lp_units = [f'{f} logprob' for f in FAM] + ['mean logprob']
display(show(summary_all, lp_units, ['C', 'M', 'M-C']))
print('Two domain experts only:')
show(summary_experts, ['mean logprob', 'mean sampled', 'mean verbal'], ['C', 'M', 'M-C'])

stat,C,M,M-C
unit,,,
gpt-4o-mini logprob,"0.64 [0.53, 0.72]","0.80 [0.68, 0.87]","0.16 [0.03, 0.27]"
gpt-4o logprob,"0.68 [0.58, 0.76]","0.89 [0.83, 0.94]","0.21 [0.11, 0.34]"
qwen3 logprob,"0.65 [0.55, 0.74]","0.87 [0.80, 0.94]","0.21 [0.13, 0.32]"
llama-3.3 logprob,"0.71 [0.62, 0.78]","0.69 [0.56, 0.86]","-0.02 [-0.11, 0.13]"
mistral-3.2 logprob,"0.68 [0.57, 0.76]","0.78 [0.66, 0.91]","0.10 [0.01, 0.22]"
gemma-3 logprob,"0.61 [0.54, 0.68]","0.71 [0.58, 0.84]","0.10 [-0.04, 0.22]"
mean logprob,"0.66 [0.58, 0.72]","0.79 [0.73, 0.87]","0.13 [0.06, 0.20]"


Two domain experts only:


stat,C,M,M-C
unit,,,
mean logprob,"0.63 [0.53, 0.72]","0.80 [0.72, 0.89]","0.17 [0.06, 0.31]"
mean sampled,"0.59 [0.52, 0.65]","0.78 [0.70, 0.88]","0.19 [0.09, 0.31]"
mean verbal,"0.49 [0.40, 0.59]","0.45 [0.34, 0.64]","-0.04 [-0.18, 0.17]"


## Step 4 — Robustness checks (Appendix C)

### 4a. Types of contestation, and what the model's mistakes consist of

A contested reading is either a **label split** or the same label with a hesitant rating; for the latter,
the annotator recorded whether the doubt concerned the *category*, the *span boundary* or *relevance*.
For each type we compute the AUROC against easy readings (log-probabilities, mean and range over models)
and the share auto-accepted at a .90 confidence threshold (Appendix Table *Contested readings by type*).

We then look at the mislabelled readings: how often is the model's answer an argument label
(`Organisation`, `Location`, a `Time` label), which our span instruction asks for whenever a span names an
actor or a place? And how well does doubt pick out the remaining mistakes, where the model chose another
event type?

In [13]:
base = df[(df.stratum == 'normative') & df.ftype.isin(['agreed', 'contested'])].copy()
c = base[base.ftype == 'contested']
base['kind'] = np.where(base.ftype == 'agreed', 'agreed',
                        np.where(base.divergent == 1, 'label split',
                                 'same label, doubt: ' + base.doubt_source.fillna('not recorded').astype(str)))
print(base[base.ftype == 'contested'].kind.value_counts().to_string(), '\n')

# log-probability runs with the confidence used for thresholds
R = []
for fam, meas, x in runs:
    if meas == 'logprob':
        x = x.merge(base[KEYS + ['kind']], on=KEYS).merge(scores(fam, 'logprob', ['consistency']), on=KEYS)
        R.append((fam, x))

def over_models(fn):
    v = [fn(x) for _, x in R]
    return np.nanmean(v), np.nanmin(v), np.nanmax(v)

rows = []
for kind in ['label split', 'same label, doubt: category', 'same label, doubt: span boundary', 'same label, doubt: relevance']:
    a = over_models(lambda x: auc(x[(x.group == 'easy') | (x.kind == kind)].doubt.values,
                                  (x[(x.group == 'easy') | (x.kind == kind)].kind == kind).values))
    rows.append({'type': kind, 'readings': (base.kind == kind).sum(), 'C (mean)': f'{a[0]:.2f}', 'C (range)': f'{a[1]:.2f}-{a[2]:.2f}',
                 'auto-accepted at .90': f"{over_models(lambda x: (x[x.kind == kind].consistency >= .9).mean())[0]:.0%}"})
for grp in ['contested', 'mislabelled', 'easy']:
    rows.append({'type': f'all {grp}', 'readings': '', 'C (mean)': '', 'C (range)': '',
                 'auto-accepted at .90': f"{over_models(lambda x: (x[x.group == grp].consistency >= .9).mean())[0]:.0%}"})
display(pd.DataFrame(rows))

comp = over_models(lambda x: x[x.group == 'mislabelled'].pred_label.isin(ARG).mean())
org = over_models(lambda x: (x[x.group == 'mislabelled'].pred_label == 'Organisation').mean())
print(f'mislabelled readings where the model chose Organisation/Location/Time: {comp[0]:.0%} (Organisation alone: {org[0]:.0%})')
def m_event(x):
    y = x[(x.group == 'easy') | ((x.group == 'mislabelled') & ~x.pred_label.isin(ARG | {NONE}))]
    return auc(y.doubt.values, (y.group == 'mislabelled').values)
me = over_models(m_event)
n_me = np.mean([((x.group == 'mislabelled') & ~x.pred_label.isin(ARG | {NONE})).sum() for _, x in R])
print(f'M for event-type mistakes only: {me[0]:.2f} (models {me[1]:.2f}-{me[2]:.2f}), about {n_me:.0f} readings per model')

kind
label split                         62
same label, doubt: span boundary    25
same label, doubt: category         18
same label, doubt: relevance        18
same label, doubt: not recorded      1 



,type,readings,C (mean),C (range),auto-accepted at .90
0,label split,62,0.71,0.65-0.74,67%
1,"same label, doubt: category",18,0.67,0.58-0.73,68%
2,"same label, doubt: span boundary",25,0.60,0.52-0.73,81%
3,"same label, doubt: relevance",18,0.57,0.47-0.70,78%
4,all contested,,,,72%
5,all mislabelled,,,,51%
6,all easy,,,,88%


mislabelled readings where the model chose Organisation/Location/Time: 54% (Organisation alone: 43%)
M for event-type mistakes only: 0.86 (models 0.71-0.92), about 16 readings per model


### 4b. Subsets: per language, and the two domain experts

English and French texts were annotated by the two domain experts, Dutch texts by a domain expert and the
trained annotator. The per-language groups are small, so we give point estimates only. The expert-pair row
with its interval comes from Step 3.

In [14]:
rows = []
for L in ['en', 'fr', 'nl']:
    v = []
    for _, x in R:
        y = x[x.lang == L]
        a = y[y.group != 'mislabelled']; b = y[y.group != 'contested']
        v.append((auc(a.doubt.values, (a.group == 'contested').values), auc(b.doubt.values, (b.group == 'mislabelled').values),
                  (y.group == 'contested').sum(), (y.group == 'mislabelled').sum()))
    v = np.array(v)
    rows.append({'language': L, 'C': f'{v[:, 0].mean():.2f}', 'M': f'{v[:, 1].mean():.2f}', 'M-C': f'{np.mean(v[:, 1] - v[:, 0]):+.2f}',
                 'contested readings': int(v[0, 2]), 'mislabelled (mean per model)': f'{v[:, 3].mean():.0f}'})
pd.DataFrame(rows)

,language,C,M,M-C,contested readings,mislabelled (mean per model)
0,en,0.57,0.75,+0.18,20,15
1,fr,0.69,0.91,+0.22,21,5
2,nl,0.66,0.75,+0.09,83,23


### 4c. Counting each passage once

The main analysis counts readings, so a passage contributes one reading per annotator, and a passage can
in principle contribute a contested reading (from the annotator who hesitated) and an agreed one (from the
partner). Here we collapse each passage into one unit and call it contested as soon as the annotators chose
different labels **or either of them** hesitated. Doubt and confidence are averaged over the passage's
readings; an agreed passage counts as mislabelled when the model is wrong on at least half of its readings.
The bootstrap resamples texts (500 samples).

In [15]:
# hesitancy of the partner annotator on the same passage
part = df[['annotator', 'hesitancy', 'pid']]
max_h = part.groupby(['pid', 'annotator']).hesitancy.max().rename('h').reset_index()
def partner_hes(r):
    h = max_h[(max_h.pid == r.pid) & (max_h.annotator != r.annotator)].h
    return h.max() if len(h) else np.nan

def passage_level(x):
    x = x.copy()
    x['partner_hes'] = [partner_hes(r) for r in x[['pid', 'annotator']].itertuples()]
    g = x.groupby('pid').agg(div=('divergent', 'max'), hes=('hesitancy', 'max'), phes=('partner_hes', 'max'),
                             doubt=('doubt', 'mean'), cons=('consistency', 'mean'),
                             correct=('pred_label', lambda s: (s == x.loc[s.index, 'label']).mean()),
                             pair=('pair', 'first'), text=('text_key', 'first'))
    g['cont'] = (g['div'] == 1) | (g.hes >= 2) | (g.phes >= 2)
    g['grp'] = np.where(g.cont, 'contested', np.where(g.correct >= .5, 'easy', 'mislabelled'))
    return g

def pstats(g):
    a = g[g.grp != 'mislabelled']; b = g[g.grp != 'contested']
    passr = lambda k: (g[g.grp == k].cons >= .9).mean()
    return (auc(a.doubt.values, (a.grp == 'contested').values), auc(b.doubt.values, (b.grp == 'mislabelled').values),
            passr('contested') / (g[g.grp != 'contested'].cons >= .9).mean(), (g.grp == 'contested').sum())

G = [passage_level(x) for _, x in R]
for tag, sel in [('all annotators', None), ('domain experts', 'expert-expert')]:
    v = np.array([pstats(g if sel is None else g[g.pair == sel]) for g in G])
    print(f'{tag}: C {v[:, 0].mean():.2f}  M {v[:, 1].mean():.2f}  M-C {np.mean(v[:, 1] - v[:, 0]):+.2f}  '
          f'ratio of auto-acceptance (contested / agreed) {v[:, 2].mean():.2f}  contested passages {int(v[0, 3])}')

rng = np.random.default_rng(1)
texts = sorted(set(G[0].text))
bs = []
for _ in range(500):
    draw = rng.choice(texts, len(texts))
    vals = []
    for g in G:
        gg = pd.concat([g[g.text == t] for t in draw])
        a = gg[gg.grp != 'mislabelled']; b = gg[gg.grp != 'contested']
        vals.append((auc(a.doubt.values, (a.grp == 'contested').values), auc(b.doubt.values, (b.grp == 'mislabelled').values)))
    vals = np.array(vals)
    bs.append((np.nanmean(vals[:, 0]), np.nanmean(vals[:, 1]), np.nanmean(vals[:, 1] - vals[:, 0])))
lo, hi = np.nanpercentile(bs, 2.5, axis=0), np.nanpercentile(bs, 97.5, axis=0)
print(f'all annotators, 95% CI: C [{lo[0]:.2f}, {hi[0]:.2f}]  M [{lo[1]:.2f}, {hi[1]:.2f}]  M-C [{lo[2]:+.2f}, {hi[2]:+.2f}]')

all annotators: C 0.64  M 0.77  M-C +0.13  ratio of auto-acceptance (contested / agreed) 0.89  contested passages 82
domain experts: C 0.62  M 0.79  M-C +0.16  ratio of auto-acceptance (contested / agreed) 0.95  contested passages 28


all annotators, 95% CI: C [0.54, 0.72]  M [0.68, 0.87]  M-C [+0.02, +0.23]


## Step 5 — Model outputs for the close-read examples (§3.3)

The close reading itself is qualitative, but the labels and confidences it cites can be checked. The
examples are identified by their position in the text (`ls_real/close_read_examples.csv`: text, annotator
and character offsets, with a short name). For each example we take the whole passage, i.e. every reading
that overlaps the example span (so both annotators' spans, even when they marked the passage differently),
and show the annotators' labels, hesitancy and source of doubt. The second table gives, for every reading
and model, the answer and confidence under all three measures (log-probabilities, the majority of the ten
sampled answers, and the mean stated confidence). The span texts are shown only when the full texts are
available locally; the quotations themselves are in §3.3 of the paper.

In [16]:
examples = pd.read_csv(os.path.join(D, 'close_read_examples.csv'))
hit = df.merge(examples, on=KEYS)
sel = df[df.pid.isin(hit.pid)].merge(hit[['pid', 'example']].drop_duplicates('pid'), on='pid').sort_values(['pid', 'annotator'])
texts = load_texts()
sel['span'] = [texts[k][s:e] if texts else f'[{s}:{e}]' for k, s, e in zip(sel.text_key, sel.start, sel.end)]
display(sel[['example', 'pid', 'span', 'annotator', 'label', 'hesitancy', 'doubt_source', 'alt_label']])

all_scores = {(fam, meas): scores(fam, meas, ['pred_label', 'consistency']) for fam in FAM for meas in MEASURES}
rows = []
for _, r in sel.iterrows():
    for fam in FAM:
        out = {'example': r.example, 'span': r.span[:35], 'annotator': r.annotator, 'model': fam}
        for meas in MEASURES:
            s = all_scores[(fam, meas)]
            s = s[(s.text_key == r.text_key) & (s.annotator == r.annotator) & (s.start == r.start) & (s.end == r.end)]
            out[meas] = f'{s.pred_label.iloc[0][:24]} ({s.consistency.iloc[0]:.2f})' if len(s) else ''
        rows.append(out)
pd.DataFrame(rows)

,example,pid,span,annotator,label,hesitancy,doubt_source,alt_label
8,Sauvages,171,[274:282],Cira,Truth-seeking initiatives,2,span boundary,Truth-seeking initiatives
9,Sauvages,171,[274:309],Elke,Cultural interventions,0,None,None
10,Sauvages,188,[1068:1103],Elke,Cultural interventions,0,None,None
24,Congoville,303,[10307:10328],A3,Cultural interventions,1,None,None
23,Congoville,303,[10307:10328],Elke,Cultural interventions,0,None,None
0,regrets,583,[2173:2180],Cira,Public acknowledgements,0,None,None
1,regrets,583,[2173:2180],Elke,Apologies,0,None,None
2,fait tomber,711,[182:232],Cira,Contestation/mobilisation,2,span boundary,Contestation/mobilisation
3,fait tomber,711,[177:188],Elke,Contestation/mobilisation,2,span boundary,None of these / not redress-related
4,Exterminez,747,[2316:2344],Cira,Truth-seeking initiatives,2,relevance,Truth-seeking initiatives


,example,span,annotator,model,logprob,sampled,verbal
0,Sauvages,[274:282],Cira,gpt-4o-mini,Cultural interventions (0.98),Cultural interventions (0.90),Cultural interventions (0.92)
1,Sauvages,[274:282],Cira,gpt-4o,Cultural interventions (1.00),Cultural interventions (0.90),Cultural interventions (0.94)
2,Sauvages,[274:282],Cira,qwen3,Cultural interventions (1.00),Cultural interventions (1.00),Cultural interventions (0.95)
3,Sauvages,[274:282],Cira,llama-3.3,Cultural interventions (1.00),Cultural interventions (1.00),Cultural interventions (0.90)
4,Sauvages,[274:282],Cira,mistral-3.2,Cultural interventions (1.00),Cultural interventions (1.00),Cultural interventions (0.94)
...,...,...,...,...,...,...,...
145,muurschildering,[11499:11514],Elke,gpt-4o,Cultural interventions (1.00),Cultural interventions (1.00),Cultural interventions (0.94)
146,muurschildering,[11499:11514],Elke,qwen3,Cultural interventions (1.00),Cultural interventions (1.00),Cultural interventions (0.95)
147,muurschildering,[11499:11514],Elke,llama-3.3,Cultural interventions (1.00),Cultural interventions (1.00),Cultural interventions (0.90)
148,muurschildering,[11499:11514],Elke,mistral-3.2,Cultural interventions (1.00),Cultural interventions (1.00),Cultural interventions (0.94)


## Step 6 — Recap: where each number appears in the paper

| Paper | Number | Step |
|---|---|---|
| §2.5 | letter mass at least 99.9% for every model except Mistral (93%) | 1 |
| Table 5 | easy 181–212, mislabelled 29–60 (241 agreed), contested 124 | 2 |
| Table 6, §3.2 | logprob C .66 [.58, .72], M .79 [.73, .87], M−C +.13 [+.06, +.20]; sampled +.10; verbal −.12; CI upper bound .72 | 3 |
| Table 6, last row | contested vs. all agreed .61 [.53, .68] / .58 / .57 | 3 |
| §3.2 | within a category .57 / .55 / .54; model-independent M−C +.09 [+.01, +.18] | 3 |
| Appendix Table *RQ2 per model*; §3.2 | Llama −.02, Gemma +.10 [−.04, +.22]; experts M−C +.17 [+.06, +.31] | 3 |
| Appendix Table *Contested readings by type*; §3.2 | label splits .71, category .67, boundary .60, relevance .57; auto-accepted 67/68/81/78% | 4a |
| §3.2 | 54% of mistakes are argument labels; event-type mistakes M .86 (about 16 per model) | 4a |
| Appendix Table *Subsets* | EN .57/.75, FR .69/.91, NL .66/.75; passages .64/.77, M−C +.13 [+.02, +.23], ratio .89; experts .62/.79, .95 | 4b, 4c |
| §3.3 | model labels and confidences of the close-read examples | 5 |